In [1]:
from pyspark.sql import SparkSession
import os
import sys

In [2]:
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .master("local[*]")    
    .getOrCreate()
)

In [3]:
df_customers = spark.read.csv("olist_customers_dataset.csv", header=True, inferSchema=True)

df_customers.show(5, truncate=True)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5 rows


In [4]:
df_orders = spark.read.csv("olist_orders_dataset.csv", header=True, inferSchema=True)

df_orders.show(5, truncate=True)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [5]:
shape = df_customers.count(), len(df_customers.columns)

shape

(99441, 5)

In [6]:
df_orders.count(), len(df_orders.columns)

(99441, 8)

In [7]:
df_customers.columns

['customer_id',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state']

In [8]:
df_orders.columns

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

In [9]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [10]:
df_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [11]:
from pyspark.sql.functions import count

df_customers.count(), df_customers.distinct().count()


(99441, 99441)

In [12]:
df_orders.count(), df_orders.distinct().count()

(99441, 99441)

In [13]:
df_customers.count(), df_customers.select("customer_unique_id").distinct().count()

(99441, 96096)

In [14]:
df_orders.count(), df_orders.select("order_id").distinct().count()

(99441, 99441)

In [15]:
df_orders.select("order_status").distinct().show()


+------------+
|order_status|
+------------+
|     shipped|
|    canceled|
|    invoiced|
|     created|
|   delivered|
| unavailable|
|  processing|
|    approved|
+------------+



In [16]:
from pyspark.sql.functions import col, sum

null_counts = df_customers.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in df_customers.columns
])
null_counts.show()


+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+



In [17]:
from pyspark.sql.functions import col, sum

null_counts = df_orders.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in df_orders.columns
])
null_counts.show()


+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [18]:
from pyspark.sql.functions import col

null_percent = null_counts.select([
    (col(c)*100 / df_orders.count()).alias(c) for c in null_counts.columns
])
null_percent.show()


+--------+-----------+------------+------------------------+------------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp| order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+------------------+----------------------------+-----------------------------+-----------------------------+
|     0.0|        0.0|         0.0|                     0.0|0.1608994278014099|          1.7930229985619615|            2.981667521444877|                          0.0|
+--------+-----------+------------+------------------------+------------------+----------------------------+-----------------------------+-----------------------------+



In [19]:
df_orders = df_orders.dropna()

In [20]:
from pyspark.sql.functions import col, sum

null_counts = df_orders.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in df_orders.columns
])
null_counts.show()


+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|                0|                           0|                            0|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [21]:
df_customers.createTempView("customers")
df_orders.createTempView("orders")

In [22]:
spark.sql("""SELECT c.customer_state, count(*) AS count FROM customers AS c
          INNER JOIN orders AS o
          ON c.customer_id = o.customer_id
          GROUP BY c.customer_state""").show()

+--------------+-----+
|customer_state|count|
+--------------+-----+
|            SC| 3547|
|            RO|  243|
|            PI|  476|
|            AM|  145|
|            RR|   41|
|            GO| 1957|
|            TO|  274|
|            MT|  886|
|            SP|40489|
|            ES| 1995|
|            PB|  517|
|            RS| 5342|
|            MS|  701|
|            AL|  397|
|            MG|11352|
|            PA|  946|
|            BA| 3256|
|            SE|  335|
|            PE| 1593|
|            CE| 1278|
+--------------+-----+
only showing top 20 rows


In [23]:
df_customers.join(df_orders, df_customers.customer_id == df_orders.customer_id, "inner").groupby("customer_state").count().show()


+--------------+-----+
|customer_state|count|
+--------------+-----+
|            SC| 3547|
|            RO|  243|
|            PI|  476|
|            AM|  145|
|            RR|   41|
|            GO| 1957|
|            TO|  274|
|            MT|  886|
|            SP|40489|
|            ES| 1995|
|            PB|  517|
|            RS| 5342|
|            MS|  701|
|            AL|  397|
|            MG|11352|
|            PA|  946|
|            BA| 3256|
|            SE|  335|
|            PE| 1593|
|            CE| 1278|
+--------------+-----+
only showing top 20 rows


In [39]:
from pyspark.sql.functions import avg

avg_delivery_orders = df_customers.join(df_orders, df_customers.customer_id == df_orders.customer_id, "inner")\
    .withColumn("avg_time_purchase_delivery", (col("order_delivered_customer_date").cast("long") - col("order_purchase_timestamp").cast("long"))/86400)\
        .agg(avg("avg_time_purchase_delivery").alias("average_delivery_time_days"))

In [47]:
avg_delivery_orders.show()

+--------------------------+
|average_delivery_time_days|
+--------------------------+
|        12.556722825661275|
+--------------------------+



In [41]:
from pyspark.sql import functions as F

d = (avg_delivery_orders
    .withColumn("total_seconds", F.round(F.col("average_delivery_time_days") * F.lit(86400)).cast("long"))

    .withColumn("DD", (F.col("total_seconds") / F.lit(86400)).cast("long"))
    .withColumn("HH", ((F.col("total_seconds") % F.lit(86400)) / F.lit(3600)).cast("long"))
    .withColumn("MM", ((F.col("total_seconds") % F.lit(3600)) / F.lit(60)).cast("long"))
    .withColumn("SS", (F.col("total_seconds") % F.lit(60)).cast("long"))

    .withColumn(
        "average_delivery_time_ddhhmmss",
        F.format_string("%02d:%02d:%02d:%02d", F.col("DD"), F.col("HH"), F.col("MM"), F.col("SS"))
    )
    .drop("total_seconds", "DD", "HH", "MM", "SS")
)

d.select("average_delivery_time_days", "average_delivery_time_ddhhmmss").show(truncate=False)


+--------------------------+------------------------------+
|average_delivery_time_days|average_delivery_time_ddhhmmss|
+--------------------------+------------------------------+
|12.556722825661275        |12:13:21:41                   |
+--------------------------+------------------------------+



In [57]:
""" Export : Sauvegarde le résultat final (Dataframe nettoyé) au format Parquet (le format standard du Big Data)."""


"""(d
    .write
    .mode("overwrite")   #.partitionBy("year", "month")  Avec partitionnement (recommandé si tu as une colonne de date / pays / etc.)
    .parquet(r"C:\tmp\average_delivery_time")
)"""


'(d\n    .write\n    .mode("overwrite")   #.partitionBy("year", "month")  Avec partitionnement (recommandé si tu as une colonne de date / pays / etc.)\n    .parquet(r"C:\tmp\x07verage_delivery_time")\n)'